In [1]:
import os

import cartopy.crs as ccrs
import cmcrameri.cm as ccm
from ipywidgets import interact
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import matplotlib.pyplot as plt

import gplately

from lib.main import *

from parameters import parameters

In [2]:
# Plate model name
plate_model_name = parameters["plate_model_name"]

# Timespan for analysis
min_time = parameters["timespan"]["min"]
max_time = parameters["timespan"]["max"]
time_steps = range(min_time, max_time + 1)

# Directory for outputs
outputs_dir = parameters["outputs_dir"]

subduction_data_filename = parameters["subduction_data_filename"]
subduction_data_filename = os.path.join(outputs_dir, subduction_data_filename)

nprocs = 8

In [3]:
plate_model_dir = "plate_model"

plate_model = get_plate_reconstruction(
    model_name=plate_model_name,
    model_dir=plate_model_dir,
)

In [4]:
if os.path.isfile(subduction_data_filename):
    subduction_data = pd.read_csv(subduction_data_filename)
else:
    subduction_data = run_calculate_convergence(
        nprocs=nprocs,
        min_time=min(time_steps),
        max_time=max(time_steps),
        plate_reconstruction=plate_model,
        verbose=True,
    )

    if subduction_data_filename is not None:
        os.makedirs(outputs_dir, exist_ok=True)
        subduction_data.to_csv(subduction_data_filename, index=False)

In [5]:
gdownload = gplately.download.DataServer(plate_model_name)
coastlines, continents, COBs = gdownload.get_topology_geometries()

subduction_data_columns = subduction_data.columns.tolist()
features_plot = subduction_data_columns.copy()
features_plot.remove('lon')
features_plot.remove('lat')
features_plot.remove('age (Ma)')
features_plot.remove('subducting_plate_ID')
features_plot.remove('trench_plate_ID')

Checking whether the requested files need to be updated...
Requested files are up-to-date!


In [6]:
@interact
def show_map(time=time_steps, feature=features_plot):
    # Call the PlotTopologies object
    gplot = gplately.PlotTopologies(plate_model, coastlines, continents, COBs, time=time)
        
    subduction_data_t = subduction_data[subduction_data["age (Ma)"] == time]

    fig = plt.figure(figsize=(16, 12))
    ax = plt.axes(projection=ccrs.Mollweide(central_longitude=200))
    ax.set_facecolor('azure')

    gplot.plot_continents(ax, edgecolor='none', facecolor='tan', alpha=0.5, zorder=2)
    gplot.plot_coastlines(ax, edgecolor='none', facecolor='tan', alpha=0.7, zorder=2)
    gplot.plot_ridges(ax, color='red', alpha=0.5, zorder=3)
    gplot.plot_plate_motion_vectors(ax, spacingX=10, spacingY=10, normalise=True, alpha=0.1, zorder=4)

    sc = ax.scatter(subduction_data_t['lon'], subduction_data_t['lat'], 50, marker='.',
                    c=subduction_data_t[feature], cmap=ccm.hawaii_r, transform=ccrs.PlateCarree(), zorder=5) # cmap: Spectral_r, YlOrRd

    gplot.plot_trenches(ax, color='k', alpha=0.3, zorder=6)
    gplot.plot_subduction_teeth(ax, spacing=0.05, color='k', alpha=0.3, zorder=7)
    
    ax.gridlines(linestyle=':')
        
    fig.colorbar(sc, orientation='horizontal', shrink=0.4, pad=0.05, label=feature, extend='both')
    
    # Define custom legend handles
    custom_handles = [
        Patch(facecolor='tan', edgecolor='none', label='Continental Crust'),  # Custom handle for the filled polygon
        Line2D([0], [0], color='red', lw=2, label='Mid-Ocean Ridge')  # Custom handle for the line (ridge)
    ]

    # Add the custom legend to the plot
    ax.legend(handles=custom_handles, loc='lower left')
    
    ax.set_title(f'Subduction Zones {time} Ma')
        
    plt.show()

interactive(children=(Dropdown(description='time', options=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, …